# MIRAGE FlowPic Classification for Network Traffic Analysis
di Mario Gabriele Carofano

Questo notebook implementa una pipeline completa di **Traffic Classification** con modelli **classici** e **ibridi quantistici** sul dataset MIRAGE, dalla preparazione dei dati fino alla valutazione finale. La pipeline è stata progettata per essere **modulare**, consentendo di scegliere dinamicamente il dataset (grezzo da file .pickle o FlowPic da file .npz), sostituire facilmente i modelli e salvare i risultati in modo organizzato.

### Panoramica

Il workflow seguito è:

1. **Setup ambiente e riproducibilità**

	- Import librerie, costanti e funzioni custom.
	- Impostazione seed (`random`, `numpy`, `torch`) e scelta automatica del device (`mps` / `cuda` / `cpu`).
	- Si può configurare l'esecuzione scegliendo i valori per le costanti e i parametri di addestramento direttamente dal file `constants.py`, oppure caricando un file di configurazione `config.json` (opzione consigliata per esperimenti riproducibili).

2. **Caricamento o generazione (opzionale) del dataset**

	- Si può scegliere se utilizzare il dataset FlowPic o il dataset grezzo.
	- In caso di utilizzo del dataset FlowPic, si può scegliere se caricare dal disco un dataset FlowPic pre-calcolato o generarlo da zero utilizzando le funzioni custom implementate nel modulo `traffic_converter.py`.
	- Indipendentemente dalla scelta, il dataset finale viene identificato dalle variabili `X_raw` (biflussi di traffico) e `y_raw` (etichette), in modo che il resto della pipeline possa essere eseguito senza modifiche.

3. **Preparazione dei dati**

	- **Label Encoding** delle classi di traffico e definizione di `N_CLASSES`.
	- Riduzione opzionale della dimensione del dataset per contenere i tempi di addestramento e l'utilizzo della memoria.
	- Suddivisione del dataset in folds per **cross-validation** tramite utilizzo di `StratifiedKFold` di `sklearn`.
	- Split stratificato in **train / validation / test**.
	- **Preprocessing (solo per dataset grezzo)**
		- Supporto a più strategie: [`Log1p`, `MinMax`, `MinMax-DirPL`, `Log1p-DirPL`].
		- Aggiornamento dinamico del numero di feature in base alla strategia selezionata.
	- **Costruzione dataset PyTorch**
		- Dataset custom `MirageDataset` (riordino tensori per CNN1D) o `FlowPicDataset` (per CNN2D).
		- Creazione di `DataLoader` per train, validation e test.

4. **Model selection**

	- Selezione dinamica del modello da `MODEL_REGISTRY`, con supporto a modelli classici e modelli ibridi quantistici (es. `AmplitudeEmbedding`, `AngleEmbedding`, `TrafficCNN`, ecc.).
	- Possibilità di recuperare un modello pre-addestrato da checkpoint.

5. **Fase di training**

	- Possibilità di scegliere l'ottimizzatore e il learning rate scheduler.
	- Possibilità di scegliere la funzione di loss.
	- Tracking metriche (per epoca).
	- Salvataggio best model con **early stopping** opzionale.

6. **Fase di valutazione (su test set)**

	- Calcolo di **accuracy**, **confusion matrix** e **classification report**.
	- I risultati sono stampati in console e salvati in cartelle create dinamicamente secondo il seguente formato: `../results/<timestamp>/<selected_fold>/`.
	- Salvataggio del file di configurazione dell'esperimento (`config.json`), contenente tutti i parametri utilizzati per l'esperimento (sia da valori scelti manualmente sia da valori generati dinamicamente).

### Dataset grezzo (configurabile)

- **Formato**: NumPy arrays serializzati in file .pickle
- **Feature per pacchetto**: 4 (DIR, PL, TCPWIN, IAT)
- **Pacchetti massimi per flusso**: 36 (o 100)
- **Pacchetti considerati**: 100 (configurabile)
- **Indicatore di padding**: -1

### Dataset di istogrammi FlowPic (configurabile)

- **Formato**: `Tuple[np.ndarray, pd.DataFrame]`, dove:
	- `np.ndarray`: Dataset di istogrammi 2D delle sessioni di traffico, di shape `(N, 1, D, D)`, dove `N` è il numero di finestre temporali valide e `D` è la dimensione dell'istogramma, calcolata in base ai parametri `MTU` e `BIN_SIZE`.
	- `pd.DataFrame`: Metadati dei flussi validi, con le colonne `"FlowID"`, `"DatasetID"` e `"Label"`.
- **Shape dell'input**: `(N, 1, D, D)` — istogrammi 2D FlowPic delle sessioni di traffico.
- **Dimensione dell'istogramma (`D`)**: Calcolata in base alle costanti `MTU` e `BIN_SIZE`.

### Soft output

Il soft output di un modello di classificazione è l'insieme di informazioni continue e probabilistiche prodotte prima della decisione finale: il vettore di confidenza associato a ciascuna classe per ogni biflusso. È l'output "grezzo" del modello che preserva l'incertezza della predizione.

-	`predictions_test.csv`
	File CSV contenente id del biflusso, etichetta ground truth, etichetta predetta, vettore di confidenza completo (una colonna per classe); viene generato per il test set.
	> *È essenziale per analisi successive come la calibrazione, il ranking delle predizioni o l'applicazione di soglie di decisione personalizzate.*
	
-	`training_history.csv`
	Storico di training / validation (loss, accuracy, tempi per epoca).

-	`calibration_report.txt`
	Rapporto di calibrazione del modello, che quantifica quanto la confidenza del modello rispecchia la sua reale accuratezza. Contiene metriche come l'Expected Calibration Error (ECE), Brier score e altro.

### Hard output

L'hard output è la decisione discreta finale derivata dal soft output tramite una regola di decisione (es. `argmax`). Corrisponde all'etichetta di classe effettivamente assegnata al biflusso e viene utilizzato per calcolare le metriche di valutazione aggregate. L'unico caso da gestire esplicitamente è il pareggio tra classi con probabilità identica, risolvibile con una convenzione di tie-breaking

> *l'hard output, essendo una proiezione con perdita di informazione, non permette di ricostruire il vettore di confidenza originario.*

-	`metrics.json`
	Versione strutturata e machine-readable delle metriche aggregate per fold. Contiene precision, recall, F1-score e support per classe, calcolati confrontando `y_pred` con `y_true`.

-	`train_val_plots.png`
	Grafici di andamento di loss e accuracy.

-	`confusion_matrix.png`
	Matrice di confusione normalizzata sul test set, costruita sulle etichette predette.

-	`reliability_diagram.png`
	Visualizzazione diretta della calibrazione. Il grafico mostra la confidenza media vs accuracy osservata per bin di confidenza.

-	`model_comparison_table.csv`
	Tabella di confronto tra modelli su fold diversi, con F1-score e accuracy, più media e deviazione standard per modello. Viene costruita aggregando i risultati nei file `metrics.json` di tutti i fold.

### Altri output

Dalla pipeline di esecuzione di questo notebook, vengono generati altri artefatti che non rientrano nella definizione di soft/hard output. Sono utili per la riproducibilità degli esperimenti, ma in effetti non sono risultati di classificazione:

-	`model.pth`
	Pesi del modello migliore (in base alla validation loss), salvati in formato PyTorch. Possono essere caricati successivamente per valutazioni o ulteriori addestramenti.

-	`config.json`
	Contiene la configurazione utilizzata per il training del modello, inclusi iperparametri, e versioni delle librerie. Utile per riprodurre esperimenti o confrontare configurazioni diverse.


---

## Setup ambiente e riproducibilità

In [ ]:
#	LIBRARIES
#   ####################################################################    #

# Importing all the constants
import importlib
import sys

sys.path.insert(1, '../src/')
import constants
importlib.reload(constants)
from constants import \
	_json_default, \
	build_session_config, \
	get_value_from_config

# Dataset loading and saving
from pathlib import Path
import pickle
import os
from traffic_converter import \
	mirage_pickle_converter, \
	flow_debug, \
	get_flowpic_dir

# Manipulation and analysis
import numpy as np
import pandas as pd

# Preprocessing and evaluation
from preprocessing_functions import *
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Machine Learning and Quantum ML
from model_selection import *
from training_functions import \
	train_epoch, \
	evaluate
from training_functions import \
	build_optimizer, \
	build_scheduler, \
	build_loss_function
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pennylane as qml

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from visualization_functions import \
	get_plot_epoch_ticks, \
	get_plot_metric_ticks, \
	draw_train_val_plot

# Other utilities
import copy
import datetime
import json
import random
import time

In [ ]:
#	FUNCTIONS
#   ####################################################################    #

def load_constants(
		names: list[str],
		use_saved_config: bool = False,
		config: dict = None,
		defaults: dict = None,
		namespace: dict = None
	) -> None:
	"""Carica i valori per le costanti specificate.

	Args:
		names (list[str]): Nomi delle costanti da caricare.
		use_saved_config (bool, optional): Indica se utilizzare
		la configurazione salvata. Defaults to False.
		config (dict, optional): Dizionario di configurazione salvata. Viene
		utilizzato solo se use_saved_config è True. Defaults to None.
		defaults (dict, optional): Dizionario dei valori di default per le costanti.
		Utilizzato solo se la costante non è presente nella fonte selezionata.
		Defaults to None.
		namespace (dict, optional): Namespace di destinazione.
		Se None, viene utilizzato il namespace globale. Defaults to None.
	
	Raises:
		KeyError: Se una o più costanti richieste non sono presenti
		nel namespace o nella configurazione salvata.
	"""

	#   ################################################################    #
	#	Inizializzazione

	namespace = namespace if namespace is not None else globals()
	defaults = defaults or {}

	#   ################################################################    #
	#	Selezione della fonte: config.json o constants.py

	print("Sorgente selezionata per il caricamento delle costanti:", end=" ")
	if use_saved_config and config is not None:
		source = config
		print("config.json")
	else:
		importlib.reload(constants)
		source = vars(constants)
		print("constants.py")

	#   ################################################################    #
	#   Recupero dei valori, con fallback ai valori di default

	missing = [n for n in names if n not in source]
	if missing:
		raise KeyError(f"Costanti non trovate nella fonte selezionata: {missing}")

	for name in names:
		namespace[name] = get_value_from_config(
			source,
			key=name,
			default=defaults.get(name)
		)

		print(f"Costante '{name}' caricata con valore: {namespace[name]}")

		# end for

	print()

	# end


In [ ]:
#	SESSION CONFIGURATION
#   ####################################################################    #

# Caricamento delle costanti strutturali del notebook.
load_constants([
	'__DEBUG',
	'__USE_PRECOMPUTED_DATASET',
	'__USE_CONFIG_FILE',
	'__CONFIG_ID',
	'__EXEC_MODE_TRAIN',
	'__SELECTED_FOLD',
	'__SAVE_OUTPUT',
])

# Si utilizza questo dizionario per salvare i valori
# utilizzati nel notebook, in modo da poterli salvare
# in un file JSON alla fine dell'esecuzione.
config = {}


In [ ]:
#	MACROS
#   ####################################################################    #

load_constants([
	'RANDOM_SEED',
	'TEST_FOLDS'
], __USE_CONFIG_FILE, config)

#   ####################################################################    #
#	Inizializzazione del timestamp per il salvataggio / caricamento
#	dei risultati e del modello.

if not __USE_CONFIG_FILE:
	id = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
	config["session_id"] = id

	output_dir = Path(f"../results/{id}/")
	output_dir.mkdir(parents=True, exist_ok=True)

	config_file_path = output_dir / "config.json"
	config_file_path.parent.mkdir(parents=True, exist_ok=True)

elif __USE_CONFIG_FILE:
	output_dir = Path(f"../results/{__CONFIG_ID}/")
	if not output_dir.exists():
		raise FileNotFoundError(f"Directory dei risultati non trovata: {output_dir}")
	
	config_file_path = output_dir / "config.json"
	if not config_file_path.exists():
		raise FileNotFoundError(f"File di configurazione non trovato: {config_file_path}")
	
	with open(config_file_path, "r", encoding="utf-8") as f:
		config = json.load(f)

	id = get_value_from_config(config, "session_id", None)
	if id and id != __CONFIG_ID:
		raise ValueError(
			f"ID della sessione nel file di configurazione ({id}) "
			f"non corrisponde a quello specificato ({__CONFIG_ID})."
		)

	# end __USE_CONFIG_FILE

#   ####################################################################    #
# 	Creazione delle directory dei risultati per il fold selezionato.

assert 0 <= __SELECTED_FOLD < TEST_FOLDS, \
    f"__SELECTED_FOLD deve essere compreso tra 0 e {TEST_FOLDS-1}."

fold_results_dir = output_dir / f"fold{__SELECTED_FOLD}"

if __EXEC_MODE_TRAIN:
	fold_results_dir.mkdir(parents=True, exist_ok=True)

elif not __EXEC_MODE_TRAIN:
	if not fold_results_dir.exists():
		raise FileNotFoundError(
			f"Directory dei risultati per il fold {__SELECTED_FOLD} "
			f"non trovata: {fold_results_dir}"
		)

	# end __USE_CONFIG_FILE

# fold_summary_path = fold_results_dir / "summary.txt"
fold_history_path = fold_results_dir / "training_history.csv"
fold_model_path = fold_results_dir / "model.pth"
fold_plots_path = fold_results_dir / "train_val_plots.png"
fold_confusion_matrix_path = fold_results_dir / "confusion_matrix.png"
fold_report_path = fold_results_dir / "classification_report.txt"

#   ####################################################################    #
# 	Configurazione del seed per la riproducibilità.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED) # Utile per hash di dizionari/set

#   ####################################################################    #
# 	Configurazione di PyTorch (CPU e GPU)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

#   ####################################################################    #
# 	Configurazione di PyTorch per la riproducibilità su GPU (CUDNN)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

#   ####################################################################    #
# 	Configurazione del dispositivo (CPU o GPU)
if not __USE_CONFIG_FILE:
	if torch.backends.mps.is_available():
		DEVICE = torch.device("mps")
	elif torch.cuda.is_available():
		DEVICE = torch.device("cuda")
	else:
		DEVICE = torch.device("cpu")

	config["device"] = DEVICE.type

elif __USE_CONFIG_FILE:
	dev_name = get_value_from_config(config, "device", "cpu")
	DEVICE = torch.device(dev_name)

	# end __USE_CONFIG_FILE
	
print(f"Using device: {DEVICE.type}")
if DEVICE.type == "cuda":
	print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
#	CLASSES
#	####################################################################    #

class MirageDataset(Dataset):
	""" Custom PyTorch Dataset per il caricamento e
	la preparazione dei dati di traffico di rete.
	
	Trasforma i dati da formato (N, Packets, Features)
	a formato (N, Features, Packets) per compatibilità con
	reti neurali convoluzionali 1D (CNN1D).
	"""
		
	def __init__(
			self, X_raw, y_raw,
			device, dtype=None, transform=None,
			lazy_device_transfer=True
		):
		"""Inizializza il dataset Mirage.

		Args:
			X_raw (torch.Tensor): Dati di input, shape (N, Packets, Features).
			y_raw (torch.Tensor): Etichette corrispondenti, shape (N,).
		"""

		# MPS non supporta i tensori float64.
		self.device = device
		self.dtype = dtype or (
			torch.float32 if device.type == "mps" else torch.float64
		)

		self.lazy_device_transfer = lazy_device_transfer
		self.transform = transform

		self.X = torch.tensor(X_raw, dtype=self.dtype).permute(0, 2, 1).to(self.device)
		self.y = torch.LongTensor(y_raw).to(self.device)

		# end

	def __len__(self):
		"""Ritorna la lunghezza del dataset.

		Returns:
			int: Lunghezza del dataset.
		"""

		return len(self.y)
	
		# end

	def __getitem__(self, idx):
		"""Ritorna un elemento del dataset.

		Args:
			idx (int): Indice dell'elemento da recuperare.

		Returns:
			tuple: Coppia (X, y) dell'elemento.
		"""

		x = self.X[idx]
		y = self.y[idx]

		if self.transform is not None:
			x = self.transform(x)
		
		return x, y
	
		# end
	
	# end class

class FlowPicDataset(Dataset):
	""" Custom PyTorch Dataset per il caricamento e la preparazione di istogrammi
    2D FlowPic (traffico di rete) e dei relativi metadati.

	A differenza della versione per sequenze di pacchetti (pensata per
    CNN1D), qui i dati sono già nel formato immagine (N, C, H, W)
    adatto a CNN2D senza alcuna trasposizione (o permutazione).

	Caratteristiche principali:
    -	Encoding automatico delle etichette testuali (Label) tramite
	sklearn.LabelEncoder, riusabile tra split train/val/test.
    -	Trasferimento "lazy" dei tensori sul device per singolo batch,
	per evitare Out-Of-Memory su GPU/MPS con dataset di grandi
	dimensioni (es. 93300 x 1 x 150 x 150 ~ 8-17 GB in RAM).
    -	Accesso ai metadati originali (FlowID, DatasetID, Label) per
	ogni campione, utile per debug/analisi degli errori.
    -	Supporto opzionale per trasformazioni (normalizzazione, augmentation).
	"""
		
	def __init__(
			self, X_raw, y_raw,
			device, dtype=None, transform=None,
			lazy_device_transfer=True
		):
		"""Inizializza il dataset FlowPic.

		Args:
			X_raw (np.ndarray): Istogrammi 2D, shape (N, 1, H, W).
			y_raw (np.ndarray): Etichette già codificate numericamente, shape (N,).
			device (torch.device): Device usato per il trasferimento lazy dei batch.
			dtype (torch.dtype, optional): Tipo dei tensori immagine.
			Se None, float32 su MPS (che non supporta float64),
			float64 altrove. Defaults to None.
			transform (Callable, optional): Funzione applicata a ciascun tensore
			immagine in __getitem__ (es. normalizzazione). Defaults to None.
			lazy_device_transfer (bool, optional): Se True, i dati restano in CPU
			(pinnati se CUDA) e vengono spostati sul device solo per il batch
			richiesto, evitando OOM su GPU/MPS. Defaults to True.
		"""

		super().__init__()

		if len(X_raw) != len(y_raw):
			raise ValueError(
				f"X_raw ({len(X_raw)}) e y_raw ({len(y_raw)}) devono avere "
				f"lo stesso numero di campioni."
			)

		# MPS non supporta i tensori float64.
		self.device = device
		self.dtype = dtype or (
			torch.float32 if device.type == "mps" else torch.float64
		)

		self.lazy_device_transfer = lazy_device_transfer
		self.transform = transform

		X_t = torch.from_numpy(np.asarray(X_raw)).to(self.dtype).contiguous()
		y_t = torch.from_numpy(np.asarray(y_raw)).long()

		if self.lazy_device_transfer:
			# Tiene i dati in CPU; il trasferimento avviene per batch in __getitem__.
			self.X = X_t.pin_memory() if self.device.type == "cuda" else X_t
			self.y = y_t
		else:
			# Trasferisce tutto sul device (attenzione a OOM su GPU/MPS).
			self.X = X_t.to(self.device, non_blocking=True)
			self.y = y_t.to(self.device, non_blocking=True)

		# end

	def __len__(self):
		"""Ritorna la lunghezza del dataset.

		Returns:
			int: Numero di campioni nel dataset.
		"""

		return self.y.shape[0]
	
		# end

	def __getitem__(self, idx):
		"""Ritorna un elemento del dataset, spostato sul device se
        richiesto dal trasferimento lazy.

		Args:
			idx (int): Indice dell'elemento da recuperare.
		
		Returns:
			tuple: Coppia (X, y) del campione richiesto.
		"""
	
		x = self.X[idx]
		y = self.y[idx]

		if self.transform is not None:
			x = self.transform(x)

		if self.lazy_device_transfer:
			x = x.to(self.device, non_blocking=True)
			y = y.to(self.device, non_blocking=True)

		return x, y

		# end
	
	# end class

---

## Caricamento o generazione (opzionale) del dataset

In [ ]:
load_constants([
	'USE_FLOWPIC_DATASET',
	'DATA_PATH',
	'DATASET_NAME',
	'TRAFFIC_FILTERS',
], __USE_CONFIG_FILE, config)

#   ####################################################################    #

# Si impostano le variabili di debug per il caricamento del dataset.
debug_cycle = False
flows_to_inspect = None

#   ####################################################################    #

print("Loading dataset...")
print(f"DEBUG impostato su {__DEBUG}.")
print(f"USE_FLOWPIC_DATASET impostato su {USE_FLOWPIC_DATASET}.")
print(f"USE_PRECOMPUTED_DATASET impostato su {__USE_PRECOMPUTED_DATASET}.\n")

if not USE_FLOWPIC_DATASET:

	# Il file pickle contiene due oggetti salvati in sequenza:
	# 1. X_raw : i dati numerici dei flussi di traffico, in formato numpy array.
	# 2. y_raw : le etichette corrispondenti ai flussi.
	# Ogni chiamata a pickle.load() legge il successivo oggetto nel file.
	with open(f"{DATA_PATH}/{DATASET_NAME}", "rb") as f:
		X_raw = np.array(pickle.load(f), dtype=np.float32)
		y_raw = np.array(pickle.load(f))

	if __DEBUG:
		example_sample = 7000
		print("Example Sample (First 5 packets):\n", X_raw[example_sample][:5])
		print("Example Label:", y_raw[example_sample])

elif USE_FLOWPIC_DATASET:

	flowpics_name = DATASET_NAME.split('.')[0]
	flowpics_dir = get_flowpic_dir(str(DATA_PATH))
	os.makedirs(flowpics_dir, exist_ok=True)

	npz_path = os.path.join(flowpics_dir, f"{flowpics_name}.npz")
	meta_path = os.path.join(flowpics_dir, f"{flowpics_name}_metadata.csv")

	if not __USE_PRECOMPUTED_DATASET:

		histograms, metadata = mirage_pickle_converter(
			f"{DATA_PATH}/{DATASET_NAME}",
			TRAFFIC_FILTERS, __DEBUG, debug_cycle, flows_to_inspect
		)

		# Se il flag di debug è attivo,
		# si stampano informazioni dettagliate sui flussi specificati.
		flow_debug(__DEBUG, flows_to_inspect, metadata, histograms)

		# Salvataggio degli istogrammi 2D FlowPic in formato NumPy.
		np.savez_compressed(npz_path, data=histograms)
		print(f"\nSalvato dataset (shape={histograms.shape}) in: {npz_path}")

		# Salvataggio dei metadati in formato CSV.
		metadata.to_csv(meta_path, index=False)
		print(f"Salvati metadati (di {len(metadata)} righe) in: {meta_path} ")

	elif __USE_PRECOMPUTED_DATASET:
		if not os.path.exists(npz_path):
			raise FileNotFoundError(f"Array di istogrammi precomputati non trovato: {npz_path}")
		if not os.path.exists(meta_path):
			raise FileNotFoundError(f"DataFrame di metadati precomputati non trovato: {meta_path}")

		# Caricamento degli istogrammi 2D FlowPic precomputati.
		histograms = np.load(npz_path)['data']
		print(f"Caricato dataset (shape={histograms.shape}) da: {npz_path}")

		# Caricamento dei metadati in formato CSV.
		metadata = pd.read_csv(meta_path)
		print(f"Caricati metadati (di {len(metadata)} righe) da: {meta_path} ")

		# end if "__USE_PRECOMPUTED_DATASET"

	# Prima di procedere con split/training, si verifica che gli istogrammi e i metadati siano allineati.
	assert histograms.shape[0] == len(metadata), "Disallineamento tra istogrammi e metadati!"
	assert list(metadata["DatasetID"]) == list(range(len(metadata))), "DatasetID non contiguo!"

	# Per coerenza con l'altro notebook, si rinominano le variabili per il dataset e le etichette.
	X_raw = histograms
	y_raw = metadata["Label"].values

	# end if "USE_FLOWPIC_DATASET"

---

## Preparazione dei dati

### Label Encoding

In questo blocco di codice, le etichette categoriche vengono trasformate in valori numerici utilizzando `LabelEncoder`. Ogni classe di traffico viene associata a un identificativo numerico, facilitando l'utilizzo nei modelli ML.

Le classi originali vengono memorizzate in `DATASET_CLASSES` per riferimenti futuri, mentre il numero totale di classi viene salvato in `N_CLASSES`.

In [ ]:
le = LabelEncoder()

y_encoded = le.fit_transform(y_raw)

N_CLASSES = len(le.classes_)
""" Number of unique classes in the dataset. """

if not __USE_CONFIG_FILE:
	config["num_classes"] = N_CLASSES

elif __USE_CONFIG_FILE:
	# Se il file di configurazione esiste, si verifica che il numero di classi
	# corrisponda a quello salvato nel file.
	config_num_classes = get_value_from_config(config, "num_classes", None)
	
	if config_num_classes != N_CLASSES:
		raise ValueError(
			f"Il numero di classi nel dataset ({N_CLASSES}) non corrisponde "
			f"a quello salvato nel file di configurazione ({config_num_classes})."
		)

	# end __USE_CONFIG_FILE

print(
    f"Number of classes: {N_CLASSES} \n\n" +
	f"Classes: {le.classes_}"
)

### K-fold Cross-Validation e suddivisione Train-Validation-Test

Il dataset viene gestito mediante una procedura di **K-fold Cross-Validation** stratificata, con K = `TEST_FOLDS`. Le proporzioni di training, validation e test sono definite dalle costanti `TRAIN_SIZE`, `VAL_SIZE` e `TEST_SIZE` *(la cui somma deve essere esattamente 1.0)*.
> *Il metodo `train_test_split` di `sklearn` viene utilizzato per garantire che la suddivisione sia stratificata, preservando la distribuzione delle classi in ciascun sottoinsieme.*

Se il flag `USE_NEW_SIZE` è attivo e il numero di campioni supera `NEW_DATASET_SIZE`, il dataset viene preliminarmente ridotto tramite campionamento casuale per contenere i tempi di addestramento, preservando la distribuzione delle classi e mantenendo la riproducibilità grazie a `RANDOM_SEED`.

Per l’esecuzione di un singolo esperimento:
- Si seleziona il fold indicato da `__SELECTED_FOLD`.

- Gli indici corrispondenti vengono usati per separare il dataset in una porzione temporanea (X_temp, y_temp) e un test set (X_test, y_test).

- La porzione temporanea viene ulteriormente suddivisa in training set (X_train, y_train) e validation set (X_val, y_val) secondo le proporzioni definite dalle costanti `TRAIN_SIZE` e `VAL_SIZE`.


In [ ]:
load_constants([
	'TEST_FOLDS',
	'TRAIN_SIZE', 'VAL_SIZE',
	'USE_NEW_SIZE', 'NEW_DATASET_SIZE',
	'RANDOM_SEED'
], __USE_CONFIG_FILE, config)

#   ####################################################################    #

# Verifica che le proporzioni di Train e Val sommino a "1".
assert TRAIN_SIZE + VAL_SIZE == 1.0, \
	"Le percentuali di Train e Val devono sommare a 1.\n" \
	f"Train: {TRAIN_SIZE:.4f}, " \
	f"Val: {VAL_SIZE:.4f}, " \
	f"Total: {(TRAIN_SIZE + VAL_SIZE):.4f}"

# Se il flag USE_NEW_SIZE è attivo, si riduce il dataset a NEW_DATASET_SIZE campioni.
if USE_NEW_SIZE and len(X_raw) > NEW_DATASET_SIZE:
	X_raw, _, y_encoded, _ = train_test_split(
		X_raw, y_encoded,
		train_size=NEW_DATASET_SIZE,
		stratify=y_encoded,
		random_state=RANDOM_SEED
	)

	# end if USE_NEW_SIZE

#   ####################################################################    #
#	Implementazione della k-fold cross-validation

skf = StratifiedKFold(
	n_splits = TEST_FOLDS,
	shuffle = True,
	random_state = RANDOM_SEED
)

folds = list(skf.split(X_raw, y_encoded))

if __DEBUG:
	print("[DEBUG] Composizione dei folds generati da StratifiedKFold:")
	for i, (train_index, test_index) in enumerate(folds):
		print(f"Fold {i}:")
		print(f"  Train: index={train_index}, length={len(train_index)}")
		print(f"  Test:  index={test_index}, length={len(test_index)}")


In [ ]:
load_constants([
	'TRAIN_SIZE', 'VAL_SIZE',
	'RANDOM_SEED',
], __USE_CONFIG_FILE, config)

#   ####################################################################    #

# Selezione del fold specificato.
train_index, test_index = folds[__SELECTED_FOLD]

# Suddivisione del training set in Train e Test set.
X_temp, X_test = X_raw[train_index], X_raw[test_index]
y_temp, y_test = y_encoded[train_index], y_encoded[test_index]

# Suddivisione del training set in Train e Validation set
val_fraction_of_temp = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=val_fraction_of_temp,
    stratify=y_temp,
    random_state=RANDOM_SEED
)

# Verifica della lunghezza dei set risultanti rispetto al dataset originale.
assert len(X_train) + len(X_val) + len(X_test) == len(X_raw), \
	"La somma dei campioni Train, Val e Test non corrisponde al totale!"

print(
    f"Selected fold: {__SELECTED_FOLD} \n" +
    f"Total samples: {len(X_raw)} \n" +
	f"Train shape: {X_train.shape} ({len(X_train)/len(X_raw):.1%}) \n" +
	f"Val shape:   {X_val.shape}   ({len(X_val)/len(X_raw):.1%}) \n" +
	f"Test shape:  {X_test.shape}  ({len(X_test)/len(X_raw):.1%})"
)

### Preprocessing (solo per dataset grezzo)

In questa sezione vengono raccolte le principali strategie di preprocessing applicate ai biflussi del dataset prima della fase di training. L'obiettivo è trasformare i dati grezzi in una rappresentazione più adatta ai modelli classici e ibridi quantistici. Queste strategie consentono di confrontare diverse modalità di preparazione dei dati, da approcci più semplici e generici a soluzioni più mirate al dominio del traffico di rete.

Le feature originali considerate sono:

- `DIR`: direzione del pacchetto
- `PL`: packet length
- `TCPWIN`: finestra TCP
- `IAT`: inter-arrival time

Le strategie di preprocessing implementate sono:
	
1. **Masking del padding + Log1p normalization:** (`Log1p`)

	Approccio guidato dal dominio applicativo. I pacchetti di padding vengono identificati e azzerati per evitare che influenzino il modello. Successivamente viene applicata una trasformazione `Log1p` alle feature numeriche (`PL`, `TCPWIN`, `IAT`) per comprimere il range dinamico e ridurre l'effetto degli outlier.

2. **Min-Max Scaling standard:** (`MinMax`)

	Approccio generico che applica una normalizzazione lineare nell'intervallo `[0, 1]` a tutte le feature. Non gestendo esplicitamente il padding, questa strategia può trattare i valori fittizi come dati reali e risultare sensibile agli outlier.

3. **Fusione DIR/PL + Min-Max Scaling:** (`MinMax-DirPL`)

	La direzione (`DIR`) e la lunghezza (`PL`) vengono combinate in una singola feature con segno, così da rappresentare il traffico in modo più compatto. Dopo questa trasformazione, il numero di feature passa da 4 a 3 e viene applicato un Min-Max Scaling standard.

4. **Masking del padding + Log1p + fusione DIR/PL:** (`Log1p-DirPL`)

	Strategia ibrida che unisce i vantaggi della Strategy 1 e della Strategy 3. Prima gestisce correttamente il padding e applica `Log1p`, poi combina `DIR` e `PL` in una feature con segno. In questo modo si ottiene una rappresentazione più compatta, semanticamente coerente e generalmente più robusta.

In [ ]:
load_constants([
    'N_PACKETS'
], __USE_CONFIG_FILE, config)

#   ####################################################################    #

# Per motivi di compatibilità, si inizializza a None.
# Si aggiornerà successivamente solo se il preprocessing
# viene applicato al dataset grezzo (Mirage) e non agli istogrammi FlowPic. 
num_features = None

print(f"USE_FLOWPIC_DATASET impostato su {USE_FLOWPIC_DATASET}.\n")
if USE_FLOWPIC_DATASET:
	print("La fase di preprocessing è disponibile solo per il dataset grezzo (Mirage).")
	pass

elif not USE_FLOWPIC_DATASET:

	PREPROCESSING_REGISTRY = {
		"Log1p": log1pPreprocessing,
		"MinMax": minMaxPreprocessing,
		"MinMax-DirPL": lambda X: minMaxPreprocessing(X, combine_dir_pl_flag=True),
		"Log1p-DirPL": lambda X: log1pPreprocessing(X, combine_dir_pl_flag=True),
	}
	""" Specifies the preprocessing strategy to apply to the dataset. """

	# Si definisce la strategia di preprocessing da applicare ai dati.
	if not __USE_CONFIG_FILE:
		PREPROCESSING_STRATEGY = "MinMax"
		config["preprocessing_strategy"] = PREPROCESSING_STRATEGY

	elif __USE_CONFIG_FILE:
		PREPROCESSING_STRATEGY = get_value_from_config(
			config,
			"preprocessing_strategy", None
		)
		
		# end __USE_CONFIG_FILE

	if PREPROCESSING_STRATEGY not in PREPROCESSING_REGISTRY:
		raise ValueError(
			f"PREPROCESSING_STRATEGY deve essere uno tra: " + 
			f"{list(PREPROCESSING_REGISTRY.keys())}"
		)

	preprocess_fn = PREPROCESSING_REGISTRY[PREPROCESSING_STRATEGY]
	print(f"Applying preprocessing strategy {PREPROCESSING_STRATEGY}...")

	X_train_proc = preprocess_fn(X_train, N_PACKETS)
	X_val_proc = preprocess_fn(X_val, N_PACKETS)
	X_test_proc = preprocess_fn(X_test, N_PACKETS)

	# Aggiorna il numero di feature in base alla strategia scelta.
	num_features = X_train_proc.shape[2]

	print("\nPreprocessing complete.")

	print(f"\nTrain shape: {X_train_proc.shape}")
	print(f"Val shape:   {X_val_proc.shape}")
	print(f"Test shape:  {X_test_proc.shape}")

	print(f"\nAdjusted number of features: {num_features}")


In [ ]:
# TODO: aggiungere applicazione transform per il dataset FlowPic.

### Costruzione dataset PyTorch

Un `DataLoader` è un oggetto che preleva campioni dal dataset e genera batch in modo efficiente.

Gli iperparametri principali del `DataLoader` sono:
-	**Batch size** (cioè il numero di campioni in un mini-batch).

	Utilizzando la GPU, un batch size più grande rende l'addestramento più efficiente. Tuttavia, un batch size più piccolo può portare a risultati migliori in termini di accuratezza finale.
	***La selezione del batch size appropriato e di altri iperparametri è fondamentale per l'ottimizzazione del modello e dipende dalle caratteristiche specifiche del dataset e dell'architettura del modello.***

-	**Number of workers**

	Questo iperparametro determina il numero di processi paralleli utilizzati per caricare i dati. Un numero maggiore di worker può accelerare il caricamento dei dati, ma può anche aumentare l'uso della memoria e la complessità del sistema.
	***È buona norma impostarli in base al numero di core della CPU.***

-	**Shuffle**

	Se impostato su `True`, i dati vengono mescolati ad ogni epoca, garantendo che il modello non impari sequenze specifiche dei dati. Si può impostare questo iperparametro per evitare overfitting e migliorare la generalizzazione del modello (solo per la fase di training).

-	**Drop last**

	Se impostato su `True`, l'ultimo batch viene scartato se non contiene il numero completo di campioni. Questo può essere utile per garantire che tutti i batch abbiano la stessa dimensione, semplificando l'addestramento del modello.
	***Si consiglia di impostarlo su `True` per il training e su `False` per la validazione e il test, perché in questi ultimi casi è importante valutare il modello su tutti i campioni disponibili.***

In [ ]:
load_constants([
    'BATCH_SIZE',
], __USE_CONFIG_FILE, config)

#   ####################################################################    #

print(f"USE_FLOWPIC_DATASET impostato su {USE_FLOWPIC_DATASET}.\n")
if USE_FLOWPIC_DATASET:
	# Si utilizza la classe dedicata FlowPicDataset per creare i dataset di PyTorch.
	train_dataset = FlowPicDataset(X_train, y_train, device=DEVICE)
	val_dataset = FlowPicDataset(X_val, y_val, device=DEVICE)
	test_dataset = FlowPicDataset(X_test, y_test, device=DEVICE)

elif not USE_FLOWPIC_DATASET:
	# Si utilizza la classe dedicata MirageDataset per creare i dataset di PyTorch,
	# in modo che siano compatibili con le reti neurali convoluzionali 1D (CNN1D).
	train_dataset = MirageDataset(X_train_proc, y_train, device=DEVICE)
	val_dataset = MirageDataset(X_val_proc, y_val, device=DEVICE)
	test_dataset = MirageDataset(X_test_proc, y_test, device=DEVICE)
	
#   ####################################################################    #

__USE_PIN_MEMORY = (DEVICE.type == "cuda")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
	num_workers=0,
    shuffle=True, drop_last=True,
    pin_memory=__USE_PIN_MEMORY,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
	num_workers=0,
    shuffle=False, drop_last=False,
    pin_memory=__USE_PIN_MEMORY,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
	num_workers=0,
    shuffle=False, drop_last=False,
    pin_memory=__USE_PIN_MEMORY,
)

---

## Model Selection

In base alla modalità di esecuzione selezionata (`__EXEC_MODE_TRAIN`), il modello viene istanziato con i parametri appropriati.

-	**Fase di training** (`__EXEC_MODE_TRAIN` impostato su `True`):
	
	Il modello viene addestrato da zero. 

-	**Fase di test** (`__EXEC_MODE_TRAIN` impostato su `False`):

	Il modello viene caricato da un file di pesi precedentemente addestrato e salvato. Inoltre, viene anche caricato il suo storico di addestramento. Il modello viene istanziato con i parametri appropriati, e i pesi salvati vengono caricati nel modello.

I modelli disponibili sono definiti nei seguenti file, e comprendono sia modelli classici che modelli ibridi quantistici:

### da models/classic_models.py

-	**Dense Model** (`ClassicalDenseBaseline`)

	Un modello classico basato su una rete neurale fully connected, progettato per elaborare input di dimensioni fisse e catturare relazioni complesse tra le feature, grazie a strati densi e funzioni di attivazione non lineari.

-	**Classical 1D CNN Model** (`TrafficCNN`)

	Un modello classico basato su una rete neurale convoluzionale 1D, progettata per estrarre caratteristiche locali dai dati sequenziali. Questo approccio sfrutta strati convoluzionali e di pooling per catturare pattern temporali o spaziali nei dati di input.

-	**Classical Twin Model** (`ClassicalTwin`)

	Un modello classico basato su una rete neurale a due rami, progettata per elaborare due flussi di dati paralleli e combinare le informazioni estratte per migliorare le prestazioni predittive.

-	**Classical Light Model** (`ClassicalLight`)

	Un modello classico leggero, progettato per essere efficiente in termini di risorse computazionali e memoria, pur mantenendo buone prestazioni predittive.

### da models/quantum_models.py

-	**Amplitude Embedding Model** (`AmplitudeEmbedding`)

	Una rete neurale ibrida che sfrutta l'Amplitude Embedding per codificare i dati classici nelle ampiezze dello stato quantistico. Questo approccio massimizza la densità dei dati, consentendo la codifica di $2^N$ features in $N$ qubit, preceduta da uno strato denso classico attivato da una funzione sigmoide.

-	**Angle Embedding Model** (`AngleEmbedding`)

	Un modello ibrido semplificato che utilizza l'Angle Embedding, dove le feature di input vengono mappate direttamente sugli angoli di rotazione dei qubit in un rapporto 1:1. Presenta uno strato di pre-elaborazione classico seguito da un circuito quantistico con strati fortemente entangled, offrendo una strategia di embedding shallow e resistente al rumore.

-	**Ring Model** (`RingEmbedding`)

	Un'architettura ibrida che impiega una strategia di Ring Embedding personalizzata, suddividendo l'input in due set di feature codificate tramite rotazioni e schemi di entanglement circolari CNOT. Questo design raddoppia la capacità dei dati rispetto al semplice angle embedding e introduce correlazioni tra i qubit già dalle prime fasi del circuito.

-	**Waterfall Model** (`WaterfallEmbedding`)

	Un modello ibrido complesso che presenta uno schema di Waterfall Embedding, suddividendo gli input in blocchi di rotazione Y e Z. Integra una connettività densa e all-to-all di porte CNOT nella prima fase, creando uno stato altamente entangled prima degli strati del variational ansatz.

-	**AmpCnn Model** (`AmpCnn`)

	Un modello ibrido che combina l'Amplitude Embedding con una rete neurale convoluzionale 1D, sfruttando le capacità di codifica quantistica per migliorare l'estrazione delle caratteristiche locali dai dati sequenziali.

-	**CnnAmpCnn Model** (`CnnAmpCnn`)

	Un modello ibrido **CNN–Quantum–CNN**: una prima CNN 1D estrae feature locali dai pacchetti, che vengono proiettate (con layer denso + sigmoide) nello spazio richiesto dall’**Amplitude Embedding**. L’output del circuito quantistico (con strati fortemente entangled) viene poi raffinato da una seconda CNN 1D e da layer fully connected per la classificazione finale multiclasse.

-	**AmpeCNNLSTMModel** (`AmpeCNNLSTMModel`)

	Un modello ibrido **CNN–Quantum–LSTM**: una prima CNN 2D estrae feature locali dai pacchetti, che vengono proiettate con layer denso + sigmoide nello spazio richiesto dall’**Amplitude Embedding**. L’output del circuito quantistico (con strati fortemente entangled) viene poi raffinato da un LSTM bidirezionale e da layer fully connected per la classificazione finale multiclasse.

### da models/flowpic_models.py

-	**FlowPicCNN Model** (`FlowPicCNN`)

	Un modello classico basato su una rete neurale convoluzionale 2D, progettata per elaborare immagini di istogrammi FlowPic. Questo approccio sfrutta strati convoluzionali e di pooling per catturare pattern spaziali nei dati di input, ottimizzando la classificazione del traffico di rete.

-	**ResNet** (`ResNetModel`)

	Questo tipo di modello introduce connessioni residue per mitigare il problema del vanishing gradient e consentire l'addestramento di reti profonde. La struttura è stata modificata per gestire istogrammi di dimensioni ridotte tipici della rappresentazione FlowPic L'architettura culmina in strati fully-connected per la classificazione finale multiclasse.

In [ ]:
load_constants([
	'N_QUBITS', 'N_LAYERS_RESNET', 'N_LAYERS_QUANTUM',
	'N_PACKETS',
	'N_SHOTS',
	'RANDOM_SEED',
], __USE_CONFIG_FILE, config)

if not __USE_CONFIG_FILE:
	SELECTED_MODEL = "ResNetModel"
	config["selected_model"] = SELECTED_MODEL

elif __USE_CONFIG_FILE:
	SELECTED_MODEL = get_value_from_config(config, "selected_model", None)
	
	# end __USE_CONFIG_FILE

#   ####################################################################    #

HybridModel = get_model_class(SELECTED_MODEL)
num_layers = N_LAYERS_RESNET if SELECTED_MODEL == "ResNetModel" else N_LAYERS_QUANTUM

model = HybridModel(
	N_QUBITS, num_layers,
	N_PACKETS, num_features,
	N_CLASSES, N_SHOTS,
	RANDOM_SEED
)

#   ####################################################################    #

# Si sposta il modello sul dispositivo corretto (CPU, GPU o MPS)
# e si imposta il tipo di dato appropriato.
model = model.to(DEVICE).float() if DEVICE.type == 'mps' else model.to(DEVICE).double()

# Inizializzazione dei pesi del modello.
model.apply(weight_init)

print(f"EXEC_MODE_TRAIN impostato su {__EXEC_MODE_TRAIN}.\n")
if __EXEC_MODE_TRAIN:
	total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
	print(f"Totale parametri addestrabili: {total_params:,}")

elif not __EXEC_MODE_TRAIN:
	print(f"Caricamento modello per la valutazione da: {fold_results_dir}")
	print("Il modello non verrà addestrato, ma solo valutato sul test set.\n")

	# Si caricano i pesi salvati del modello dalla directory di output specificata.
	weights = torch.load(fold_model_path, map_location=DEVICE)

	# Verifica la compatibilità tra il modello corrente e i pesi salvati.
	check_model_compatibility(model, weights)

	# Carica i pesi nel modello corrente e stampa l'esito del caricamento.
	load_result = model.load_state_dict(weights)
	print(f"\nEsito caricamento: {load_result}")
	print("Model loaded correctly.\n")

	# Si imposta il modello in modalità di valutazione
	# per disabilitare il dropout e la batch normalization.
	model.eval()

	# Si recupera lo storico delle metriche di addestramento e validazione dal file CSV salvato.
	df_history = pd.read_csv(fold_history_path)
	history = df_history.to_dict(orient='list')

	# end if

print(f"Modello selezionato: {SELECTED_MODEL} -> {HybridModel.__name__}")
print(model.get_model_name())
if __DEBUG: print(model)

---

## Fase di training

La fase di training è il processo iterativo mediante il quale il modello apprende i **parametri (pesi)** ottimali a partire dai dati di addestramento. Ad ogni iterazione, il modello elabora un batch di dati, calcola le predizioni, ne misura la discrepanza rispetto ai target reali tramite la **loss function**, e aggiorna i propri pesi tramite **backpropagation** e **optimizer**. Questo ciclo si ripete per un numero prefissato di epoche.

> *La scelta e la configurazione dei seguenti iperparametri ha un impatto diretto sulla velocità di convergenza, sulla stabilità dell'addestramento e sulla capacità del modello di generalizzare su dati non visti.*

### Scelta dell'optimizer

È l’algoritmo che aggiorna i pesi del modello dopo ogni batch, usando i gradienti calcolati con la backpropagation (e.g. in pratica decide come e quanto modificare i parametri per ridurre l’errore). Pertanto, la scelta dell’optimizer (e del **learning rate**) influenza stabilità, velocità di convergenza e qualità finale delle predizioni.

Le scelte più comuni sono:

- `SGD`

	Ottimizzatore “classico”: aggiorna i pesi con passi uniformi (spesso con `momentum`).  
	È adatto quando si vuole un comportamento semplice e controllabile, e funziona bene su modelli/dataset stabili (con tuning accurato del learning rate).

- `Adam`

	In generale, `Adam` è preferito come scelta di default perché usa learning rate adattivi e converge più rapidamente su problemi complessi o con poco tuning.

- `RMSprop`, `AdamW`

	`RMSprop` è spesso utile con gradienti rumorosi/non stazionari (es. dati sequenziali), mentre `AdamW` è indicato quando si vuole una regolarizzazione migliore grazie al weight decay disaccoppiato.

In [ ]:
load_constants([
	'LEARNING_RATE', 'WEIGHT_DECAY',
	'EPSILON', 'MOMENTUM'
], __USE_CONFIG_FILE, config)

if not __USE_CONFIG_FILE:
	SELECTED_OPTIMIZER = "AdamW"
	config["selected_optimizer"] = SELECTED_OPTIMIZER

elif __USE_CONFIG_FILE:
	SELECTED_OPTIMIZER = get_value_from_config(
		config,
		"selected_optimizer", None
	)
	
	# end __USE_CONFIG_FILE

#   ####################################################################    #

if not __EXEC_MODE_TRAIN:
	print(f"EXEC_MODE_TRAIN impostato su False.\n")
	print("La selezione dell'optimizer è irrilevante "
	   "in questa modalità.")


### Scelta dello scheduler

È un meccanismo che modifica dinamicamente il learning rate durante l’addestramento, in base a una strategia predefinita. L’obiettivo è migliorare la convergenza e la stabilità del training, evitando di rimanere bloccati in minimi locali o oscillazioni eccessive.

Le strategie disponibili includono:

- `OneCycleSched`

	Implementa la strategia **One Cycle Policy**, che aumenta il learning rate fino a un massimo e poi lo riduce, con un ciclo di warm-up e cool-down. Questo approccio può accelerare la convergenza e migliorare le prestazioni finali del modello.

- `LinearSched`

	Implementa una riduzione lineare del learning rate durante l’addestramento, partendo da un valore iniziale e scendendo fino a un valore finale. È utile quando si desidera un controllo più semplice e prevedibile sul learning rate.

- `StepSched`

	Implementa una riduzione a gradini del learning rate, diminuendolo di un fattore `gamma` specificato ogni `step_size` epoche. Questo approccio può essere utile quando si desidera un controllo più granulare sul learning rate e si sospetta che il modello possa beneficiare di periodi di apprendimento più lento.

- `NoSched`

	Non applica alcuna modifica al learning rate durante l’addestramento, mantenendo il valore iniziale costante.

In [ ]:
load_constants([
	'EPOCHS',
	'MAX_LR',
	'START_FACTOR',
	'STEP_SIZE', 'STEPLR_GAMMA'
], __USE_CONFIG_FILE, config)

if not __USE_CONFIG_FILE:
	SELECTED_SCHEDULER = "OneCycleSched"
	config["selected_scheduler"] = SELECTED_SCHEDULER
	
elif __USE_CONFIG_FILE:
	SELECTED_SCHEDULER = get_value_from_config(
		config,
		"selected_scheduler", None
	)
	
	# end __USE_CONFIG_FILE

#   ####################################################################    #

if not __EXEC_MODE_TRAIN:
	print(f"EXEC_MODE_TRAIN impostato su False.\n")
	print("La selezione dello scheduler è irrilevante "
	   "in questa modalità.")


### Scelta della loss function (`criterion`)

È la funzione che misura quanto le predizioni del modello sono lontane dai target reali. Fornisce il segnale da minimizzare durante l’addestramento.

Nel problema di classificazione multiclasse spesso si usano:

- `CrossEntropy`

	È la scelta standard per la **classificazione multiclasse**, che misura la differenza tra la distribuzione di probabilità predetta e la distribuzione reale delle etichette. Penalizza fortemente le predizioni molto sicure ma sbagliate, guidando il modello a migliorare le sue predizioni minimizzando questa differenza durante l'addestramento. È adatta quando il dataset è **abbastanza bilanciato** o quando non si vuole introdurre un trattamento diverso tra classi.

- `WeightedCrossEntropy`

	Stessa idea della CrossEntropy standard, ma con `weight=class_weights`. Assegna pesi diversi a ciascuna classe in base alla loro frequenza nel training set. Le classi con meno campioni ricevono pesi più alti, aiutando il modello a prestare maggiore attenzione durante l'addestramento. È indicata per dataset con uno **sbilanciamento moderato**.

- `Focal`

	La Focal Loss è progettata per affrontare dataset **fortemente sbilanciati**, riducendo il peso degli esempi facili e concentrandosi maggiormente su quelli difficili o mal classificati. È utile quando il modello tende a favorire troppo le classi più frequenti. I pesi vengono calcolati nello stesso modo della Weighted CrossEntropy.
	
	È necessario definire due parametri: `alpha` serve a bilanciare le classi, mentre `gamma` è un parametro di focalizzazione regolabile che determina quanto velocemente gli esempi facili vengono ridotti di peso.

In [ ]:
load_constants([
	'ALPHA', 'FOCAL_LOSS_GAMMA'
], __USE_CONFIG_FILE, config)

if not __USE_CONFIG_FILE:
	SELECTED_LOSS = "WeightedCrossEntropy"
	config["selected_loss"] = SELECTED_LOSS
elif __USE_CONFIG_FILE:
	SELECTED_LOSS = get_value_from_config(
		config,
		"selected_loss", None
	)
	
	# end __USE_CONFIG_FILE

#   ####################################################################    #

if not __EXEC_MODE_TRAIN:
	print(f"EXEC_MODE_TRAIN impostato su False.\n")
	print("La selezione della loss function è irrilevante "
	   "in questa modalità.")


### Training loop

Dopo aver configurato l’ottimizzatore e la loss function, il training procede per un numero definito di epoche. In ogni epoca, il modello viene addestrato sui batch del training set e valutato sul validation set. 

Le principali metriche monitorate durante il training sono:

-	**Loss**:
	
	Misura l’errore del modello. L’obiettivo è minimizzare questa metrica.

-	**Accuracy**:

	Misura la percentuale di predizioni corrette. L'obiettivo è massimizzare questa metrica, ma può essere meno informativa in presenza di dataset sbilanciati.

-	**Time**:

	Misura il tempo impiegato per completare l'epoca, utile per stimare la durata complessiva del training e per ottimizzare la configurazione degli iperparametri.

In [ ]:
load_constants([
	'EPOCHS', 'PATIENCE', 'EARLY_STOPPING'
], __USE_CONFIG_FILE, config)

#   ####################################################################    #

if not __EXEC_MODE_TRAIN:
	print(f"EXEC_MODE_TRAIN impostato su False.\n")
	print("Il modello non verrà addestrato.")
	pass

elif __EXEC_MODE_TRAIN:
	# Questo dizionario terrà traccia delle metriche
	# di addestramento e validazione per ogni epoca.
	history = {
		'epoch': [],
		'time': [],
		'accuracy': [],
		'val_accuracy': [],
		'loss': [],
		'val_loss': [],
	}

	# Inizializza le variabili per il monitoraggio della miglior loss di validazione.
	best_val_loss = float('inf')
	best_model_wts = copy.deepcopy(model.state_dict())
	patience_counter = 0

	#   ####################################################################    #
	#	SCELTA DEGLI IPERPARAMETRI PER IL TRAINING

	# Scelta dell'optimizer
	optimizer = build_optimizer(
		SELECTED_OPTIMIZER, model,
		LEARNING_RATE, WEIGHT_DECAY,
		EPSILON, MOMENTUM
	)

	# Scelta del learning rate scheduler
	lr_sched = build_scheduler(
		SELECTED_SCHEDULER, optimizer, train_loader,
		EPOCHS,
		MAX_LR,
		START_FACTOR,
		STEP_SIZE, STEPLR_GAMMA
	)

	# Scelta della loss function
	criterion = build_loss_function(
		SELECTED_LOSS, y_train, N_CLASSES, DEVICE,
		ALPHA, FOCAL_LOSS_GAMMA
	)

	#   ####################################################################    #
	#	INIZIO TRAINING LOOP

	start_time = time.time()

	print(
		f"Addestramento iniziato.\n"
		f"[DATETIME] {id}\n"
		f"[MODEL] {model.get_model_name()}\n"
			f"{SELECTED_LOSS} | "
			f"{SELECTED_OPTIMIZER} | "
			f"{SELECTED_SCHEDULER}\n"
		f"[DATASET] {DATASET_NAME}\n"
			f"Training samples: {len(train_loader.dataset)} | "
			f"Validation samples: {len(val_loader.dataset)} | "
			f"Batch size: {BATCH_SIZE}\n"
	)

	for epoch in range(EPOCHS):
		
		# Salva il timestamp di inizio epoca per calcolare la durata dell'epoca corrente.
		start_epoch_time = time.time()
		print(f"\nEpoch {epoch+1}/{EPOCHS}")
		
		# Calcola la loss e l'accuratezza per il training set e il validation set.
		train_loss, train_acc = train_epoch(
			model, DEVICE,
			train_loader,
			criterion, optimizer, lr_sched
		)
		
		val_loss, val_acc = evaluate(
			model, DEVICE,
			val_loader,
			criterion
		)
		
		# Calcola la durata dell'epoca corrente
		# e il tempo totale trascorso dall'inizio dell'addestramento.
		end_epoch_time = time.time()
		epoch_duration = end_epoch_time - start_epoch_time
		elapsed_time = end_epoch_time - start_time

		# Aggiorna lo storico delle metriche per l'epoca corrente.
		history['epoch'].append(epoch + 1)
		history['time'].append(epoch_duration)
		history['loss'].append(train_loss)
		history['accuracy'].append(train_acc)
		history['val_loss'].append(val_loss)
		history['val_accuracy'].append(val_acc)

		# Stampa in console le metriche dell'epoca corrente.
		print(
			f"Loss: {train_loss:.4f} - Acc: {train_acc:.2f}%\n"
			f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.2f}%\n"
			f"Epoch time: {epoch_duration:.2f}s | Elapsed time: {elapsed_time:.2f}s"
		)

		# CHECK : se la loss di validazione migliora,
		# salva il modello e resetta il contatore per la patience.
		if val_loss < best_val_loss:
			best_val_loss = val_loss
			best_model_wts = copy.deepcopy(model.state_dict())
			patience_counter = 0
			print(f"  -> Validation loss improved. Model saved.")
		else:
			patience_counter += 1
			print(f"  -> No improvement. ", end="")
			print(f"Patience: {patience_counter} of {PATIENCE}" if True else "")

		# Se il contatore di patience raggiunge il limite
		# e l'early stopping è abilitato, si interrompe il training.
		if patience_counter >= PATIENCE and EARLY_STOPPING:
			print("Early stopping triggered.")
			break

		# end for epoch

	# Calcola e stampa il tempo totale di addestramento.
	total_time = time.time() - start_time
	print(f"\nTraining complete in {total_time/60:.2f} minutes.")

	# Salva i pesi del modello con la miglior loss di validazione.
	last_model = copy.deepcopy(model)
	model.load_state_dict(best_model_wts)


### Salvataggio del modello

Questa sezione gestisce il salvataggio del modello addestrato e dei risultati del training.

Inoltre, se il flag `__USE_CONFIG_FILE` è disattivato, salva anche la configurazione corrente dei valori costanti, variabili e di ambiente in un file JSON. In questo modo, è possibile riprodurre l'esperimento in futuro o confrontare le prestazioni tra diverse configurazioni.

In [ ]:
if not __EXEC_MODE_TRAIN:
	print(f"EXEC_MODE_TRAIN impostato su False.\n")
	print("Non è stato eseguito alcun addestramento. Il modello non verrà salvato.")
	pass

elif __EXEC_MODE_TRAIN:

	#   ####################################################################    #
	#	Salvataggio dei risultati del fold corrente

	# with open(fold_summary_path, "w") as f:
	# 	model.summary(print_fn=lambda x: f.write(x + "\n"))

	# Salva lo storico delle metriche di addestramento e validazione in un file CSV.
	df_history = pd.DataFrame(history)
	df_history.to_csv(fold_history_path, index=False)
	print(f"Storico delle metriche salvato in {fold_history_path}")

	# Salva i pesi del modello addestrato in un file .PTH
	torch.save(model.state_dict(), fold_model_path)
	print(f"Modello salvato in {fold_model_path}")

	#   ####################################################################    #
	#	Salvataggio del file di configurazione in formato JSON

	if not __USE_CONFIG_FILE:
		final_session_config = build_session_config(constants, config)

		with open(config_file_path, "w", encoding="utf-8") as file:
			json.dump(
				final_session_config,
				file, indent=4,
				default=_json_default,
				ensure_ascii=False
			)

		print(f"\nFile di configurazione salvato in {config_file_path}")

		# end if


---

## Fase di valutazione

Dopo la fase di training, il modello viene valutato sul test set per misurare le sue prestazioni su dati mai visti prima.

Le principali metriche calcolate includono:

- **Loss**:

	Misura l’errore del modello sul test set.
	
- **Accuracy**:

	Misura la percentuale di predizioni corrette sul test set.

Si completa il notebook con la visualizzazione dei risultati finali e la generazione di report dettagliati. Questi includono grafici della funzione di loss e di accuracy durante le fasi di training e validazione, la matrice di confusione normalizzata sul test set, e un report di classificazione con precision, recall, F1-score e support per ciascuna classe.

### Valutazione dell'accuracy sul test set

In [ ]:
print(f"EXEC_MODE_TRAIN impostato su {__EXEC_MODE_TRAIN}.\n")
if not __EXEC_MODE_TRAIN:
	print("Non è stato eseguito alcun addestramento.\n"
		"Il modello da valutare è stato caricato da un checkpoint salvato.")
	test_loss, test_acc = evaluate(model, DEVICE, test_loader)

elif __EXEC_MODE_TRAIN:
	test_loss, test_acc = evaluate(model, DEVICE, test_loader, criterion)

# Calcola la loss e l'accuratezza per il test set,
# per valutare le prestazioni del modello su dati mai visti prima.
print(f"Test Loss: {test_loss:.4f} | " if test_loss is not None else "", end="")
print(f"Test Acc: {test_acc:.2f}%")

### Grafici di loss e accuracy su training / validation set

In [ ]:
# Recupero dello storico delle metriche di addestramento e validazione.
train_loss = history['loss']
val_loss = history['val_loss']
train_acc = history['accuracy']
val_acc = history['val_accuracy']

# Calcolo dell'epoca con validation loss minima per evidenziarla nel grafico.
best_epoch = int(np.argmin(val_loss)) +1

#   ####################################################################    #

plots = plt.figure(figsize=(12, 5))

loss_plot = plt.subplot(1, 2, 1)
draw_train_val_plot(loss_plot, 'Loss', train_loss, val_loss, best_epoch)

acc_plot = plt.subplot(1, 2, 2)
draw_train_val_plot(acc_plot, 'Accuracy', train_acc, val_acc, best_epoch)

plt.tight_layout()
plt.show()

#   ####################################################################    #

if __SAVE_OUTPUT:
	plots.savefig(fold_plots_path)

### Matrice di confusione

Una matrice di confusione è una tabella che riassume le prestazioni di un modello di classificazione, mettendo a confronto le classi reali (righe) con le classi predette dal modello (colonne). Ogni cella `(i,j)` indica quante osservazioni della classe reale `i` sono state classificate come classe predetta `j`, permettendo di vedere non solo quante previsioni sono corrette, ma anche quali errori specifici il modello commette.

In [ ]:
all_preds = []
all_labels = []

# Si utilizza "torch.no_grad()" per disabilitare
# il calcolo del gradiente durante la fase di valutazione,
# migliorando le prestazioni e riducendo l'uso della memoria.
with torch.no_grad():
    for inputs, labels in test_loader:
        
		# Sposta i dati sul DEVICE selezionato prima della predizione.
        inputs = inputs.to(DEVICE)
        
		# Calcola le predizioni del modello per il batch corrente.
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        
		# Infine, si utilizza "extend" per aggiungere le predizioni e le etichette
        # del batch corrente alle liste globali "all_preds" e "all_labels".
        # A differenza di "append", che aggiunge un singolo elemento,
		# "extend" aggiunge tutti gli elementi di un Iterable alla lista esistente.
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
		# ATTENZIONE!
		# Sostituire con "all_labels.extend(labels.numpy())" se si utilizza la CPU.
    
		# end for inputs, labels

cm = confusion_matrix(all_labels, all_preds, normalize='true')

confusion_matrix_plot = plt.figure(figsize=(10, 8))

ax = sns.heatmap(
	cm,
	annot=False,
	fmt='',
	cmap='plasma_r',   # Mappa colori plasma invertita
	linewidths=0.5,    # Spessore linee della griglia
	mask= cm == 0,     # Applica la maschera
	linecolor='black', # Colore linee della griglia
	square=True,       # Celle quadrate
	cbar_kws={ "ticks": [0.1, 1, 10, 100] },
	xticklabels=le.classes_,
	yticklabels=le.classes_
)

ax.set_facecolor('white')

empty_cols = np.where(cm.sum(axis=0) == 0)[0]

# Se ci sono colonne vuote, aggiunge un simbolo '•' al centro di ciascuna cella vuota.
for col in empty_cols:
    for row in range(cm.shape[0]):
        
        # Per centrare il testo nella cella, si aggiunge 0.5 a "row" e "col".
        ax.text(
            col + 0.5, row + 0.5, '•',
            ha='center', va='center',
            color='black', fontsize=15
		)
    
		# end for row
	# end for col

plt.title("Confusion Matrix")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

if __SAVE_OUTPUT:
	confusion_matrix_plot.savefig(fold_confusion_matrix_path)

### Report di classificazione completo

In [ ]:
importlib.reload(constants)
from constants import __SAVE_OUTPUT

#   ####################################################################    #

report_dict = classification_report(
    all_labels, all_preds,
    target_names=le.classes_, labels=np.arange(N_CLASSES),
    digits=4, zero_division=0
)

print(report_dict)

if __SAVE_OUTPUT:
    with open(fold_report_path, "w") as f:
        f.write(report_dict)

---